
> ## ✅ 결론 — Optuna 튜닝: **소폭 개선 → 채택**
>
> | 모델 | baseline R² | tuned Val R² | 개선 |
> |---|---|---|---|
> | **XGB** ⭐ | 0.0824 | **0.0881** | +0.0057 |
> | CB | 0.0861 | 0.0921 | +0.0060 |
> | LGB | 0.0835 | 0.0880 | +0.0045 |
>
> - 세 모델 모두 튜닝으로 R² 소폭 상승 (+0.004~0.006).
> - 최적 파라미터 → `best_params.json` 저장, 이후 실험에서 재사용.
> - **→ 튜닝된 XGB를 본 모델 계열로 확정.**


# 10. 모델 튜닝 실험
- 확정 조건: pitch15, with delta, baseline (이상치 처리 없음)
- Optuna로 XGBoost / CatBoost / LightGBM 각 50 trials 튜닝
- paired t-test로 최종 모델 선택

| 실험 | 내용 |
|---|---|
| E5-1 | XGBoost Optuna 50 trials |
| E5-2 | CatBoost Optuna 50 trials |
| E5-3 | LightGBM Optuna 50 trials |
| E5-4 | 최적 모델 간 paired t-test (n=30 seeds) |

In [ ]:
import os, sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
else:
    DRIVE = r'c:\Users\suyou\OneDrive\Desktop\ASAC\PROJECT\투수 컨디션 예측'

FEATURE_PATH = os.path.join(DRIVE, '0_data', '4_features', 'features_pitch15.parquet')
OUTPUT_DIR   = os.path.join(DRIVE, '4_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'데이터: {FEATURE_PATH}')

In [ ]:
try:
    import optuna
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'optuna', '-q'])
    import optuna

try:
    import xgboost, catboost, lightgbm
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'catboost', 'lightgbm', '-q'])

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats
import matplotlib.pyplot as plt
import warnings, logging
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'Optuna   : {optuna.__version__}')
print(f'XGBoost  : {xgboost.__version__}')
print(f'LightGBM : {lgb.__version__}')

In [ ]:
df = pd.read_parquet(FEATURE_PATH)

META_COLS    = ['game_pk', 'pitcher', 'season', 'y_whiff']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

train = df[df['season'].isin([2021, 2022, 2023])]
val   = df[df['season'] == 2024]
test  = df[df['season'] == 2025]

X_train, y_train = train[FEATURE_COLS], train['y_whiff']
X_val,   y_val   = val[FEATURE_COLS],   val['y_whiff']
X_test,  y_test  = test[FEATURE_COLS],  test['y_whiff']

print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}  Features: {len(FEATURE_COLS)}')

## XGBoost Optuna 튜닝

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
        'early_stopping_rounds': 50,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = model.predict(X_val)
    return r2_score(y_val, pred)

xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nXGB 최적 Val R²: {xgb_study.best_value:.4f}')
print(f'최적 파라미터: {xgb_study.best_params}')

## CatBoost Optuna 튜닝

In [ ]:
def cb_objective(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 100, 1000),
        'learning_rate':       trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':               trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_seed': 42, 'verbose': False,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    pred = model.predict(X_val)
    return r2_score(y_val, pred)

cb_study = optuna.create_study(direction='maximize')
cb_study.optimize(cb_objective, n_trials=50, show_progress_bar=True)

print(f'\nCB 최적 Val R²: {cb_study.best_value:.4f}')
print(f'최적 파라미터: {cb_study.best_params}')

## LightGBM Optuna 튜닝

In [ ]:
def lgb_objective(trial):
    params = {
        'num_leaves':        trial.suggest_int('num_leaves', 16, 256),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 15),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 5.0),
        'objective': 'regression', 'metric': 'rmse',
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    n_est = trial.suggest_int('n_estimators', 100, 1000)
    lgb_tr = lgb.Dataset(X_train, label=y_train)
    lgb_vl = lgb.Dataset(X_val,   label=y_val, reference=lgb_tr)
    model  = lgb.train(
        params, lgb_tr, num_boost_round=n_est,
        valid_sets=[lgb_vl],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = model.predict(X_val)
    return r2_score(y_val, pred)

lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f'\nLGB 최적 Val R²: {lgb_study.best_value:.4f}')
print(f'최적 파라미터: {lgb_study.best_params}')

## 튜닝 결과 비교

In [ ]:
baseline = {'XGB': 0.0824, 'CB': 0.0861, 'LGB': 0.0835}
tuned    = {
    'XGB': xgb_study.best_value,
    'CB':  cb_study.best_value,
    'LGB': lgb_study.best_value,
}

print(f'{"모델":<8} {"베이스라인 R²":>14} {"튜닝 후 R²":>12} {"개선":>8}')
print('-' * 48)
for m in ['XGB', 'CB', 'LGB']:
    diff = tuned[m] - baseline[m]
    print(f'{m:<8} {baseline[m]:>14.4f} {tuned[m]:>12.4f} {diff:>+8.4f}')

best_model = max(tuned, key=tuned.get)
print(f'\n튜닝 후 최고 모델: {best_model} (Val R²={tuned[best_model]:.4f})')

## 최적 모델 간 Paired t-test (n=30 seeds)

In [ ]:
N_SEEDS = 30
r2_xgb, r2_cb, r2_lgb = [], [], []

xgb_params = {k: v for k, v in xgb_study.best_params.items()}
cb_params  = {k: v for k, v in cb_study.best_params.items()}
lgb_params = {k: v for k, v in lgb_study.best_params.items() if k != 'n_estimators'}
lgb_n_est  = lgb_study.best_params.get('n_estimators', 500)

for seed in range(N_SEEDS):
    # XGBoost
    m = xgb.XGBRegressor(**{**xgb_params,
        'random_state': seed, 'n_jobs': -1, 'verbosity': 0,
        'early_stopping_rounds': 50})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    r2_xgb.append(r2_score(y_val, m.predict(X_val)))

    # CatBoost
    m = CatBoostRegressor(**{**cb_params, 'random_seed': seed, 'verbose': False})
    m.fit(X_train, y_train, eval_set=(X_val, y_val))
    r2_cb.append(r2_score(y_val, m.predict(X_val)))

    # LightGBM
    lgb_tr = lgb.Dataset(X_train, label=y_train)
    lgb_vl = lgb.Dataset(X_val,   label=y_val, reference=lgb_tr)
    m = lgb.train(
        {**lgb_params, 'objective': 'regression', 'metric': 'rmse',
         'random_state': seed, 'n_jobs': -1, 'verbose': -1},
        lgb_tr, num_boost_round=lgb_n_est,
        valid_sets=[lgb_vl],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    r2_lgb.append(r2_score(y_val, m.predict(X_val)))

    if (seed + 1) % 10 == 0:
        print(f'seed {seed+1}/{N_SEEDS} 완료')

print('\n=== Paired t-test 결과 ===')
pairs = [('XGB vs CB', r2_xgb, r2_cb), ('XGB vs LGB', r2_xgb, r2_lgb), ('CB vs LGB', r2_cb, r2_lgb)]
for label, a, b in pairs:
    t, p = stats.ttest_rel(a, b)
    winner = 'XGB' if label.startswith('XGB') and np.mean(a) > np.mean(b) else label.split(' vs ')[1 if np.mean(a) < np.mean(b) else 0]
    print(f'{label}: mean({np.mean(a):.4f} vs {np.mean(b):.4f})  t={t:.3f}  p={p:.4f}  {"✅ 유의" if p < 0.05 else "❌ 유의하지 않음"}')

## 최종 모델 확정 및 저장

In [ ]:
import joblib

# ↓ paired t-test 결과 보고 최종 모델 선택 (기본: XGB)
FINAL_MODEL = 'XGB'

if FINAL_MODEL == 'XGB':
    final = xgb.XGBRegressor(**{**xgb_study.best_params,
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
        'early_stopping_rounds': 50})
    final.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
elif FINAL_MODEL == 'CB':
    final = CatBoostRegressor(**{**cb_study.best_params, 'random_seed': 42, 'verbose': False})
    final.fit(X_train, y_train, eval_set=(X_val, y_val))
else:  # LGB - 별도 처리 필요
    raise NotImplementedError('LGB는 아래 별도 셀에서 처리')

val_r2  = r2_score(y_val,  final.predict(X_val))
test_r2 = r2_score(y_test, final.predict(X_test))
val_rmse  = mean_squared_error(y_val,  final.predict(X_val))  ** 0.5
test_rmse = mean_squared_error(y_test, final.predict(X_test)) ** 0.5

print(f'최종 모델: {FINAL_MODEL}')
print(f'Val  RMSE={val_rmse:.4f}  R²={val_r2:.4f}')
print(f'Test RMSE={test_rmse:.4f}  R²={test_r2:.4f}')

model_path = os.path.join(OUTPUT_DIR, f'tuned_{FINAL_MODEL.lower()}_final.pkl')
joblib.dump(final, model_path)
print(f'\n모델 저장 -> {model_path}')

In [ ]:
results = pd.DataFrame({
    'model':        ['XGB', 'CB', 'LGB'],
    'baseline_r2':  [0.0824, 0.0861, 0.0835],
    'tuned_val_r2': [xgb_study.best_value, cb_study.best_value, lgb_study.best_value],
    'tuned_mean_r2': [np.mean(r2_xgb), np.mean(r2_cb), np.mean(r2_lgb)],
    'tuned_std_r2':  [np.std(r2_xgb),  np.std(r2_cb),  np.std(r2_lgb)],
})
results['improvement'] = results['tuned_val_r2'] - results['baseline_r2']

out_path = os.path.join(OUTPUT_DIR, 'tuning_results.csv')
results.to_csv(out_path, index=False)
print(results.round(4).to_string(index=False))
print(f'\n저장 완료 -> {out_path}')

In [ ]:
import json

best_params_all = {
    'FINAL_MODEL': FINAL_MODEL,
    'XGB': {k: v for k, v in xgb_study.best_params.items()},
    'CB':  {k: v for k, v in cb_study.best_params.items()},
    'LGB': {k: v for k, v in lgb_study.best_params.items()},
}

json_path = os.path.join(OUTPUT_DIR, 'best_params.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(best_params_all, f, indent=2, ensure_ascii=False)

print(f'파라미터 저장 완료 → {json_path}')
print(json.dumps(best_params_all, indent=2))